In [1]:
!pip -q install pyreadstat tabulate
import os, re
import numpy as np
import pandas as pd
import pyreadstat

from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/hdac_data'
PATH = f'{DATA_DIR}/HN24_ALL.sav'
print(os.path.exists(PATH))


df, meta = pyreadstat.read_sav(PATH)
print(df.shape)

Mounted at /content/drive
True
(6997, 813)


In [2]:
df['stress'] = np.select([df['BP1'].isin([1, 2]), df['BP1'].isin([3, 4])], [1, 0], default=np.nan)


WD = dict(bed_h='BP16_11', bed_m='BP16_12', wake_h='BP16_13', wake_m='BP16_14')   # 주중
WE = dict(bed_h='BP16_21', bed_m='BP16_22', wake_h='BP16_23', wake_m='BP16_24')   # 주말
SPECIAL = [88, 99]

def clean(col):
    return df[col].where(~df[col].isin(SPECIAL))

def sleep_hours(v):
    bed  = clean(v['bed_h'])  * 60 + clean(v['bed_m'])
    wake = clean(v['wake_h']) * 60 + clean(v['wake_m'])
    return ((wake - bed) % (24 * 60)) / 60

df['sleep_wd'] = sleep_hours(WD)
df['sleep_we'] = sleep_hours(WE)
df['sleep'] = (df['sleep_wd'] * 5 + df['sleep_we'] * 2) / 7


RAW = dict(
    sex='sex', age='age',
    marry='marri_1',      # 결혼 여부 (코드 1, 2)
    income='incm',        # 소득 4분위수
    edu='edu',            # 교육수준
    econ='EC1_1',         # 경제활동 상태
    phq9='mh_PHQ_S',      # 우울 PHQ-9 총점
    gad7='mh_GAD_S',      # 불안 GAD-7 점수
)
not_found = {k: v for k, v in RAW.items() if v not in df.columns}
if not_found:
    import re
    print("df 에 없는 변수:", not_found)
    labels = meta.column_names_to_labels
    for k in not_found:
        pat = {'marry':'결혼|혼인','income':'소득','edu':'교육','econ':'경제활동','phq9':'PHQ|우울','gad7':'GAD|불안'}.get(k, k)
        hits = [(c, labels.get(c)) for c in df.columns if re.search(pat, str(labels.get(c)) + c, re.I)]
        print(f"  [{k}] 후보:", hits[:10])
    raise KeyError("RAW 의 변수명을 위 후보로 고친 뒤 다시 실행하세요")
for new, old in RAW.items():
    if new not in ('sex', 'age'):
        df[new] = df[old]

df['in_analysis'] = (df['age'] >= 19) & df['stress'].notna() & df['sleep'].between(3, 16)
print("in_analysis n =", int(df['in_analysis'].sum()), "(Table 1 N = 5,800)")

in_analysis n = 5800 (Table 1 N = 5,800)


In [12]:
CFG = dict(
    exposure='stress',
    outcome='sleep',
    weight='wt_itvex',
    strata='kstrata',
    psu='psu',
    adj_cat=['sex', 'marry', 'income', 'edu', 'econ'],
    adj_num=['age'],
    amb_num=['phq9', 'gad7'],
)

need = [CFG[k] for k in ['exposure', 'outcome', 'weight', 'strata', 'psu']] + CFG['adj_cat'] + CFG['adj_num'] + CFG['amb_num'] + ['in_analysis']
missing_cols = [c for c in need if c not in df.columns]
assert not missing_cols, f"missing columns: {missing_cols}"
assert set(df[CFG['exposure']].dropna().unique()) <= {0, 1}, 'exposure must be 0/1'

In [13]:
from scipy import stats

def svy_ols(d, y, xcols, w, strata, psu):

    X = np.column_stack([np.ones(len(d))] + [d[c].to_numpy(float) for c in xcols])
    yy = d[y].to_numpy(float); ww = d[w].to_numpy(float)
    ok = ww > 0
    A = X[ok].T @ (X[ok] * ww[ok, None]); Ai = np.linalg.inv(A)
    beta = Ai @ (X[ok].T @ (ww[ok] * yy[ok]))
    e = np.where(ok, yy - X @ beta, 0.0)
    U = X * (ww * e)[:, None]
    key = d[strata].astype(str) + "|" + d[psu].astype(str)
    g = pd.DataFrame(U).groupby([d[strata].values, d[psu].values]).sum()   # 집락별 점수 합
    B = np.zeros_like(A); nh_tot = 0; npsu = 0
    for h, gh in g.groupby(level=0):
        n = len(gh); npsu += n; nh_tot += 1
        if n < 2: continue                      # 집락 1개 층은 분산 기여 없음(lonely PSU 제외)
        Z = gh.to_numpy() - gh.to_numpy().mean(0)
        B += n / (n - 1) * Z.T @ Z
    V = Ai @ B @ Ai
    df = npsu - nh_tot
    se = np.sqrt(np.diag(V)); t = beta / se; p = 2 * stats.t.sf(np.abs(t), df)
    q = stats.t.ppf(.975, df)
    names = ["const"] + list(xcols)
    return pd.DataFrame({"beta": beta, "se": se, "lo": beta - q*se, "hi": beta + q*se, "p": p}, index=names), int(ok.sum()), df

In [14]:
# 경제활동 상태: 코드 1, 2만 사용 (11주차와 동일)
df['econ'] = df['econ'].where(df['econ'].isin([1, 2]))

VL = getattr(meta, 'variable_value_labels', None) or {}

def special_codes(col, extra=()):
    labs = VL.get(col, {}) or {}
    pat = re.compile(r'모름|무응답|비해당|해당\s*없|응답\s*거부')
    return {k for k, v in labs.items() if pat.search(str(v))} | set(extra)

WHT23_EXTRA_SPECIAL = [999]
WHT5_EXTRA_SPECIAL = [99]

for col in ['EC_wht_23', 'EC_wht_5']:
    print(col, '| 변수 설명:', meta.column_names_to_labels.get(col) if hasattr(meta, 'column_names_to_labels') else None)
    print('   값 설명:', VL.get(col))

sp23 = special_codes('EC_wht_23', WHT23_EXTRA_SPECIAL)
sp5 = special_codes('EC_wht_5', WHT5_EXTRA_SPECIAL)
df['wht_hr'] = df['EC_wht_23'].where(~df['EC_wht_23'].isin(sp23))     # 주당 근로시간
df['wht_tm'] = df['EC_wht_5'].where(~df['EC_wht_5'].isin(sp5))        # 근로시간대 (범주)
print('결측 처리한 코드 | 근로시간:', sorted(sp23), '| 근로시간대:', sorted(sp5))

wk = df['in_analysis'] & df['econ'].eq(1)
print('분석 대상 중 경제활동 상태 = 1 (근로자):', int(wk.sum()), '명')
print('근로시간 요약(근로자):')
print(df.loc[wk, 'wht_hr'].describe().round(1).to_string())
print('근로시간 > 168 (불가능한 값) n =', int((df.loc[wk, 'wht_hr'] > 168).sum()))
print('근로시간대 분포(근로자):')
print(df.loc[wk, 'wht_tm'].value_counts(dropna=False).sort_index().to_string())

EC_wht_23 | 변수 설명: 주당 평균 근로시간
   값 설명: None
EC_wht_5 | 변수 설명: 주/변형근로시간
   값 설명: None
결측 처리한 코드 | 근로시간: [999] | 근로시간대: [99]
분석 대상 중 경제활동 상태 = 1 (근로자): 3397 명
근로시간 요약(근로자):
count    3396.0
mean       35.9
std        15.8
min         1.0
25%        25.0
50%        40.0
75%        45.0
max        98.0
근로시간 > 168 (불가능한 값) n = 0
근로시간대 분포(근로자):
wht_tm
1.0    2786
2.0     331
3.0      65
4.0     123
5.0      47
6.0      11
7.0      31
8.0       2
NaN       1


In [15]:
E, Y = CFG['exposure'], CFG['outcome']
d = df.copy()
inA = d['in_analysis'].astype(bool)
main_cat, main_num = CFG['adj_cat'], CFG['adj_num']
amb = CFG['amb_num']

def design(dd, cols_cat, cols_num):
    inside = (dd['_w'] > 0).to_numpy()
    X = pd.DataFrame({E: dd[E].astype(float)})
    for c in cols_cat:
        cats = sorted(dd.loc[inside, c].dropna().unique())
        for k in cats[1:]:
            X[f"{c}_{k}"] = (dd[c] == k).astype(float)      # 가장 작은 코드가 기준 범주
    for c in cols_num:
        X[c] = dd[c].astype(float)
    X = X.fillna(0.0)
    X[Y] = dd[Y].fillna(0.0)
    X['_w'] = dd['_w'].values
    X[CFG['strata']] = dd[CFG['strata']].values
    X[CFG['psu']] = dd[CFG['psu']].values
    return X

def fit_model(domain, cols_cat, cols_num):
    used = list(cols_cat) + list(cols_num)
    ok = domain.astype(bool)
    if used:
        ok = ok & d[used].notna().all(axis=1)                 # 완전 사례
    dd = d[[E, Y, CFG['strata'], CFG['psu']] + used].copy()
    dd['_w'] = np.where(ok, d[CFG['weight']].fillna(0), 0.0)
    X = design(dd, cols_cat, cols_num)
    xs = [c for c in X.columns if c not in (Y, '_w', CFG['strata'], CFG['psu'])]
    res, n, dof = svy_ols(X, Y, xs, '_w', CFG['strata'], CFG['psu'])
    r = res.loc[E]
    inside = X['_w'] > 0
    return dict(beta=r.beta, lo=r.lo, hi=r.hi, p=r.p, n=n, dof=dof, covars=[c for c in xs if c != E],
                n_exp=int((inside & (X[E] == 1)).sum()), n_ref=int((inside & (X[E] == 0)).sum()))

# --- 주 분석(모형 2) 다시 실행 ---
main = fit_model(inA, main_cat, main_num)
print(f"주 분석(모형 2): beta = {main['beta']:.3f} ({main['lo']:.3f}, {main['hi']:.3f}), p = {main['p']:.4f}, N = {main['n']:,}, df = {main['dof']}")
print("11주차 Table 2 모형 2: beta = -0.187 (-0.269, -0.104), N = 5,396")

same = (main['n'] == 5396 and round(main['beta'], 3) == -0.187
        and round(main['lo'], 3) == -0.269 and round(main['hi'], 3) == -0.104)
print("11주차와 같은 결과인가:", "예" if same else "아니오 -> 위 코드나 자료를 다시 확인하세요")
STRICT = globals().get('STRICT', True)
assert same or not STRICT, "주 분석이 11주차 결과와 다릅니다"
print("주 분석 보정변수:", main['covars'])

주 분석(모형 2): beta = -0.187 (-0.269, -0.104), p = 0.0000, N = 5,396, df = 165
11주차 Table 2 모형 2: beta = -0.187 (-0.269, -0.104), N = 5,396
11주차와 같은 결과인가: 예
주 분석 보정변수: ['sex_2.0', 'marry_2.0', 'income_2.0', 'income_3.0', 'income_4.0', 'edu_2.0', 'edu_3.0', 'edu_4.0', 'econ_2.0', 'age']


In [16]:
# ---------- S1. 우울·불안 추가 보정 (모형 3) ----------
s1_ref = fit_model(inA & d[amb].notna().all(axis=1), main_cat, main_num)   # 주 분석을 같은 사람으로
s1 = fit_model(inA, main_cat, main_num + amb)
assert s1['n'] == s1_ref['n'], "S1: 비교 대상 인원이 다릅니다"

# ---------- S3. 수면시간 유효 범위 4~14시간 ----------
d['in_s3'] = (d['age'] >= 19) & d['stress'].notna() & d['sleep'].between(4, 14)
assert not (d['in_s3'] & ~inA).any(), "S3 대상은 주 분석 대상 안에 있어야 합니다"
s3 = fit_model(d['in_s3'], main_cat, main_num)
n_drop_s3 = int((inA & ~d['in_s3']).sum())

# ---------- S4. 근로 관련 변수 ----------
workers = inA & d['econ'].eq(1)
s4a = fit_model(workers, main_cat, main_num)                      # 근로자만 (econ 은 한 범주뿐이라 더미가 생기지 않음)
s4b = fit_model(workers, main_cat + ['wht_tm'], main_num + ['wht_hr'])
s4b_ref = fit_model(workers & d[['wht_hr', 'wht_tm']].notna().all(axis=1), main_cat, main_num)
assert s4b['n'] == s4b_ref['n'], "S4b: 비교 대상 인원이 다릅니다"

print("S1 모형 보정변수:", s1['covars'])
print("S3: 주 분석 대상에서 빠진 인원(수면시간 3~4시간 또는 14~16시간):", n_drop_s3, "명")
print("S4a 보정변수(econ 더미 없음 확인):", s4a['covars'])
print("S4b 보정변수:", s4b['covars'])
print("S4b: 근로자 중 근로 변수 결측으로 빠진 인원:", s4a['n'] - s4b['n'], "명")
print("분석별 노출군/비교군 n:")
for name, r in [('주 분석', main), ('S1', s1), ('S3', s3), ('S4a', s4a), ('S4b', s4b)]:
    print(f"  {name}: {r['n_exp']:,} / {r['n_ref']:,}")

S1 모형 보정변수: ['sex_2.0', 'marry_2.0', 'income_2.0', 'income_3.0', 'income_4.0', 'edu_2.0', 'edu_3.0', 'edu_4.0', 'econ_2.0', 'age', 'phq9', 'gad7']
S3: 주 분석 대상에서 빠진 인원(수면시간 3~4시간 또는 14~16시간): 26 명
S4a 보정변수(econ 더미 없음 확인): ['sex_2.0', 'marry_2.0', 'income_2.0', 'income_3.0', 'income_4.0', 'edu_2.0', 'edu_3.0', 'edu_4.0', 'age']
S4b 보정변수: ['sex_2.0', 'marry_2.0', 'income_2.0', 'income_3.0', 'income_4.0', 'edu_2.0', 'edu_3.0', 'edu_4.0', 'wht_tm_2.0', 'wht_tm_3.0', 'wht_tm_4.0', 'wht_tm_5.0', 'wht_tm_6.0', 'wht_tm_7.0', 'wht_tm_8.0', 'age', 'wht_hr']
S4b: 근로자 중 근로 변수 결측으로 빠진 인원: 1 명
분석별 노출군/비교군 n:
  주 분석: 1,317 / 4,079
  S1: 1,315 / 4,071
  S3: 1,307 / 4,063
  S4a: 900 / 2,489
  S4b: 900 / 2,488


In [17]:
def fmt_ci(r):
    return f"{r['beta']:.3f} ({r['lo']:.3f}, {r['hi']:.3f})"

def fmt_p(v):
    return '<0.001' if v < 0.001 else f"{v:.3f}"

SPEC = [
    ('주 분석 (모형 2)',                    main, None),
    ('S1 우울·불안 추가 보정 (모형 3)',      s1,   s1_ref),
    ('S3 수면시간 4–14시간만',              s3,   main),
    ('S4a 근로자만',                        s4a,  main),
    ('S4b 근로자 + 근로시간·시간대 보정',    s4b,  s4b_ref),
]

rows = []
for label, r, ref in SPEC:
    rows.append({
        '분석': label,
        '분석 인원(명)': f"{r['n']:,}",
        'β (95% CI)': fmt_ci(r),
        'p': fmt_p(r['p']),
        '기준 인원(명)': '-' if ref is None else f"{ref['n']:,}",
        '기준 β (95% CI)': '-' if ref is None else fmt_ci(ref),
        '차이 Δβ': '-' if ref is None else f"{r['beta'] - ref['beta']:+.3f}",
    })
table3 = pd.DataFrame(rows).set_index('분석')
print("Table 3. 스트레스 인지(기준: 비인지)와 하루 평균 수면시간(시간): 주 분석과 민감도분석")
print(table3.to_markdown())
print()
print("기준: S1 = 주 분석을 같은 사람(우울·불안 응답자)으로 적합 / S3, S4a = 주 분석 / S4b = 주 분석을 같은 사람(근로자, 근로 변수 응답자)으로 적합")

Table 3. 스트레스 인지(기준: 비인지)와 하루 평균 수면시간(시간): 주 분석과 민감도분석
| 분석                              | 분석 인원(명)   | β (95% CI)              | p      | 기준 인원(명)   | 기준 β (95% CI)         | 차이 Δβ   |
|:----------------------------------|:----------------|:------------------------|:-------|:----------------|:------------------------|:----------|
| 주 분석 (모형 2)                  | 5,396           | -0.187 (-0.269, -0.104) | <0.001 | -               | -                       | -         |
| S1 우울·불안 추가 보정 (모형 3)   | 5,386           | -0.129 (-0.226, -0.033) | 0.009  | 5,386           | -0.185 (-0.268, -0.102) | +0.056    |
| S3 수면시간 4–14시간만            | 5,370           | -0.167 (-0.249, -0.085) | <0.001 | 5,396           | -0.187 (-0.269, -0.104) | +0.020    |
| S4a 근로자만                      | 3,389           | -0.150 (-0.248, -0.052) | 0.003  | 5,396           | -0.187 (-0.269, -0.104) | +0.037    |
| S4b 근로자 + 근로시간·시간대 보정 | 3,388           | -0.122 (-0.221, -0.023) | 0.016  | 3,388           | -0.150 (

In [18]:
summ = []
for label, r, ref in SPEC[1:]:
    summ.append({
        '분석': label,
        '방향(기준과 같은가)': '같음' if np.sign(r['beta']) == np.sign(ref['beta']) else '다름',
        '크기 변화율(%)': f"{(r['beta'] / ref['beta'] - 1) * 100:+.1f}",
        '95% CI에 0 포함': '예' if r['lo'] <= 0 <= r['hi'] else '아니오',
        'p < 0.05': '예' if r['p'] < 0.05 else '아니오',
        '기준 β가 이 분석의 95% CI 안': '예' if r['lo'] <= ref['beta'] <= r['hi'] else '아니오',
    })
print(pd.DataFrame(summ).set_index('분석').to_markdown())
print()
print("각주용 정보")
print("  가중치:", CFG['weight'], "| 층:", CFG['strata'], "| 집락:", CFG['psu'])
print("  자유도(집락 수 - 층 수):", main['dof'])
print("  분산: Taylor 선형화. 집락이 1개인 층은 분산에 기여하지 않게 처리(근사)")
print("  자료: 2024년 국민건강영양조사 (HN24_ALL.sav)")

| 분석                              | 방향(기준과 같은가)   |   크기 변화율(%) | 95% CI에 0 포함   | p < 0.05   | 기준 β가 이 분석의 95% CI 안   |
|:----------------------------------|:----------------------|-----------------:|:------------------|:-----------|:-------------------------------|
| S1 우울·불안 추가 보정 (모형 3)   | 같음                  |            -30.3 | 아니오            | 예         | 예                             |
| S3 수면시간 4–14시간만            | 같음                  |            -10.6 | 아니오            | 예         | 예                             |
| S4a 근로자만                      | 같음                  |            -19.7 | 아니오            | 예         | 예                             |
| S4b 근로자 + 근로시간·시간대 보정 | 같음                  |            -18.4 | 아니오            | 예         | 예                             |

각주용 정보
  가중치: wt_itvex | 층: kstrata | 집락: psu
  자유도(집락 수 - 층 수): 165
  분산: Taylor 선형화. 집락이 1개인 층은 분산에 기여하지 않게 처리(근사)
  자료: 2024년 국민건강영양조사 (HN24_ALL.sav)
